# Reading the ball

**A traceable walkthrough of our frame-level possession model.**

More than a Hack 2027 · Barcelona vs Athletic Club. This notebook is prose and figures only: the code that produced every number and every clip is in `solution/`, and the appendix at the bottom goes from a sentence to the exact command that recomputes it.

## 1. The problem: a ball that keeps disappearing

Watch the clip below. Barcelona is moving the ball in their own half. The players are jogging into shape, the ball is visible, and then the tracking loses it. Six seconds later the ball is still gone and the players have stopped. The answer strip under the pitch turns gray: the stretch is **dead**.

<img src="figs/fig_match_gap.gif" style="width:100%">

The clip is twenty seconds of our match, rendered from the same tracking file the submission uses. The gray band is a **ball-missing run**: a run of frames where the tracker has no ball position at all. The red band is the model's **D** label. On our match the ball is missing for 40.7% of the graded frames, and almost all of that missing time is a stoppage. That single fact is the spine of this notebook.

The task is to give one of **three labels** to every graded frame. **H** means Barcelona has the ball, **A** means Athletic has it, and **D** means the ball is dead: a throw-in, a corner, a foul, a goal, an injury, a cooling break. A **possession** is a maximal unbroken run of H or A; D ends it. We answer at 25 Hz, for 158,857 graded frames out of 199,319 in the match, and the answer has to be a coherent sequence, not a per-frame guess.

<img src="figs/fig_three_labels.png" style="width:100%">

The obvious first rule is **who is nearest the ball**. It is the organizers' own starter baseline, and it is a good place to start: with a 2.5 m radius, a player near the ball is usually the player who has it. But nearest is not **owner**. The owner is the team that last touched the ball; it keeps the ball through a pass in the air, a dribble, or a scramble, until the other team touches it. In the schematic below the defender is the nearest player to the ball, but the ball is still traveling from the passer to the receiver, so the owner has not changed.

<img src="figs/fig_nearest_player.png" style="width:100%">

This is not a corner case. On clean practice data the nearest-player rule catches .764 of the visible changes of owner; on data degraded to our noise it catches .482. About 13.5% of owner changes cannot be resolved from geometry at all: the ball is hidden, the players are in a duel, or the tracking disagrees with the label. A model that only looks at the nearest player is therefore blind to a large part of the problem, and that is why the rest of this notebook is about two other clues: **ball missingness** and **team shape**.

## 2. The score: why one extra boundary is expensive

Imagine one true possession: Barcelona has the ball for ten seconds, then the whistle goes. The animation below is synthetic — no match data — and it is scored by the organizer's own metric, not by a cartoon. It is the single most important figure in this notebook, because the scoring rule explains almost every design choice we made.

<img src="figs/fig_spellf1.gif" style="width:100%">

The score is **spellF1**. Possessions are maximal runs of H or A. Each predicted possession is matched one-to-one to a true possession by **temporal IoU**, which is the length of the shared frames divided by the length of the union: one means the two intervals are identical, zero means they do not overlap. The F-score of that matching is averaged over IoU thresholds from 0.3 to 0.7. A prediction is not credited for finding the right team if its interval is in the wrong place.

The animation walks through the same true spell five times. A perfect answer scores 1.00. Split the answer in two with a one-second D gap and the score falls to .40: each fragment has an IoU of 0.50 against the true spell, so it is still inside the averaging band, but only one fragment can be matched and the other is a false positive. Split it in three and the score is .10; each fragment's IoU is 0.33, at the edge of the band. Split it in four and the score is .00: every fragment's IoU is 0.25, below the 0.3 floor, so nothing matches. Miss the spell completely and the score is also .00.

<img src="figs/fig_spellf1_curve.png" style="width:100%">

The asymmetry is the point. A missed possession costs one false negative. A spurious boundary costs a false positive *and* damages the host possession it cut, because the host now has to compete with its own fragment for the one-to-one match. This is why we are conservative about inserting hand-overs, and why a clue that improves frame accuracy can still lose score. On a full match, differences below about 0.04 are noise. The public five minutes hold only 17 true possessions, which is why we never choose a method on them.

## 3. The practice set: ten matches, made to look like ours

We have the true answers for five public minutes of our match, and the rules forbid tuning on them. 17 possessions is not enough to choose a threshold, and the organizers show that tuning on that sample picks the wrong answer even for a one-parameter baseline. So every choice is made on other professional matches with public answers, after we make their data look like ours.

The clip below is one practice passage, before and after **degrading**. On the left, the clean match: the ball stays visible while play is stopped, so a ball-based rule would see the stoppage for free. On the right, the same passage after the ball is dropped where our tracker drops it, the ball path is smoothed, and the gap edges are jittered. The stoppage is still there in the players; the ball is not.

<div style="display:flex;"><div style="flex:1; padding:0 4px;"><center><img src="figs/fig_practice_clean.gif" style="width:100%"></center></div><div style="flex:1; padding:0 4px;"><center><img src="figs/fig_practice_degraded.gif" style="width:100%"></center></div></div>

The practice set is **10 matches**: seven from the German league and three from Metrica. A second set of 20 SkillCorner matches is used only for who-has-the-ball questions, never pooled with the ten. The ten are frozen: same games, same seeds, same degrader, same cells. Any method we compare sees exactly the same data.

<img src="figs/fig_missingness.png" style="width:100%">

The histogram is the reason the practice set had to be repaired. Our match hides the ball for 40.7% of graded frames; the degraded practice set hides it for 37.3%, close enough. The conditional level did not match at first: the original degrader dropped the ball at random, including during live play, so 42% of practice live frames had no ball against ~3% of ours. After the fix the practice set is at 4.4%, and 2.1% of long gaps are live against ≤ ~4% of ours. The old gap taught the model that a missing ball is compatible with live play, which is exactly wrong for our data.

We found two flaws and fixed both. The first was the **resting ball**: in the practice data a ball lying still for a restart shook about twenty times more than in a real match, so ball motion separated dead from live far too well. Fixing the sign dropped the texture-only separability from a perfect score to .92 on the ten games and .907 on twenty held-out SkillCorner games, still worse than our .80 target; ball-motion clues are therefore quoted as upper bounds. The second was **where the ball hides**. After the fix, Baseline+ moved from .216 to .326, and the in-gap decoder's gain fell from +.082 to +.006. Every number measured before that fix was wrong in the same direction.

One property cannot be measured on our match: how often a stoppage keeps the ball visible. Label-free instruments put it between 7% and 23% of stoppages, so every method is tested across that range. The practice set is also about .06 easier than the real match — the organizers' two-second baseline scores .3037 there and .243 on the real match — so we trust the ranking of methods, never the level.

## 4. The pipeline, one stage at a time

Here is a practice passage through the shipped stack. The first three stages use one passage; the last three use a passage where that stage actually fires, because no single passage exercises all six. Each stage has one job, one number on the frozen practice set, and one way to hurt us if it is wrong. The numbers are marginal: each stage is measured on the stack it actually ships on, not as an additive decomposition.

**Stage 1 — nearest player, smoothed.** The organizers' two-second baseline: the nearest player within a radius votes, and the vote is majority-smoothed over fifty frames. It is the floor we have to beat: .3037 spellF1 on the frozen practice set.

<img src="figs/fig_stage_smooth50.gif" style="width:100%">

**Stage 2 — hysteresis and merge (Baseline+).** A player within 2.5 m owns the ball; otherwise the last owner keeps it. Short flickers are held back and short spells are merged. On the practice set it scores .3259, a gain of +.0222 [+.0058, +.0403] over the two-second baseline. That lead does not survive a half-second shift of the answers, so the honest reading is *about level to slightly ahead*. If hysteresis is too strong, we absorb a real short turnover; if it is too weak, we emit flickers, which the score punishes twice.

<img src="figs/fig_stage_baseline_plus.gif" style="width:100%">

**Stage 3 — the missing-ball gate.** When the ball has been gone for a while and the players are standing still, the stretch becomes D. This is the largest single gain in the stack: +.0728 [+.0649, +.0808] on the practice set, with no game negative. On our match the expected gain is +.06 to +.08 because our gaps are longer and cleaner than the practice set's. If the gate is wrong, it turns live possession into a stoppage and cuts a real spell in two, which is the most expensive error the score can charge.

<img src="figs/fig_stage_gate.gif" style="width:100%">

**Stage 4 — the in-gap decoder.** Inside a missing-ball gap, team shape says which team is wider and therefore more likely to have the ball. The decoder is allowed to insert a hand-over there. On the practice set it adds +.0052 [+.0038, +.0065]; on our match only 3.9% of graded frames are eligible, so the expected gain is +.002 to +.005. The clip below shows the mechanism, not a success story: on this passage the decoder inserts an H island where the answer stays A. That is exactly why its average gain is small and why it runs only after the gate.

<img src="figs/fig_stage_ingap.gif" style="width:100%">

**Stage 5 — grown stoppages.** A resting ball is a real stoppage, but it is only a sliver of one. We extend each resting-ball moment out to the stoppage around it, with a radius and a thirty-second cap. On the practice set this adds +.0055 [.0044, .0066], expected +.001 to +.005 on our match. It is only safe on top of the gate: on raw Baseline+ most of the changed frames land in live play. If the radius is too wide, we eat the restart.

<img src="figs/fig_stage_grown.gif" style="width:100%">

**Stage 6 — the sub-gap rule.** Some stoppages sit inside missing-ball gaps that the main gate left alone. A second, weaker rule fires when the stoppage probability stays above one half for two seconds inside such a gap. It adds +.0044 [.0033, .0056] on the practice set, expected about +.002 on our match. If it fires on a live gap, it creates a D island inside a possession.

<img src="figs/fig_stage_subgap.gif" style="width:100%">

The stack is not quite the order above: the build applies the gate, then grown stoppages, then the in-gap decoder, then the sub-gap rule. The marginal numbers are the ones measured for each module on its shipping base in the build cards, so read them as evidence for each stage, not as a running sum. The next tier-1 add-on is not wired yet: deleting every H/A spell shorter than one and a half seconds and absorbing it into D adds +.0099 [.0063, .0138] on the practice set, with an expected +.006 to +.009 on our match. It is waiting for the build card.

| stage | what it does | practice spellF1 | marginal gain | expected on our match |
|---|---|---|---|---|
| smooth50 | nearest player, majority-smoothed | .3037 | — | — |
| Baseline+ | hysteresis, merge, 2.5 m radius | .3259 | +.0222 [+.0058, +.0403] | level to slightly ahead |
| gate | missing ball + still players -> D | .3987 | +.0728 [+.0649, +.0808] | +.06 to +.08 |
| in-gap decoder | team shape inside a missing-ball gap | .4039 | +.0052 [+.0038, +.0065] | +.002 to +.005 |
| grown stoppages | resting ball -> whole stoppage | — | +.0055 [.0044, .0066] | +.001 to +.005 |
| sub-gap rule | second gate inside unfired gaps | — | +.0044 [.0033, .0056] | +.002 |

The gate-plus-decoder stack is +.1003 [+.0877, +.1130] over the two-second baseline on the practice set. On our match the frozen shipped file calls 45.0% of graded frames dead, finds 414 possessions across 114 D runs, and leaves 58.3 min of ball in play. The build is the clean-clone Q26 reproduction, sha `8255d2d1…5e61`. Those are label-free descriptions, not scores.

## 5. The two clues that carry it

In the clip at the top of the notebook the ball is gone for six seconds. Two completely different instruments were asked the same question — is this missing time a stoppage? — and they agree. One is a classifier trained on player motion. The other is the broadcast camera's own ball track, which follows the ball through gaps the tactical tracker misses. Neither uses the answers.

<img src="figs/fig_gate_instruments.png" style="width:100%">

The player-motion classifier flags 85.6% [80.8%, 89.2%] of our missing time as dead; the broadcast route puts 84.5% of it dead. After correcting for the frames each instrument cannot see, the probability that a missing frame is dead is between .93 and .99. Two instruments with different failure modes landing on the same number is the strongest label-free evidence we have. It is still not a license to call every gap dead: a long live dropout would cut a possession, so the gate also requires the players to be still.

The second clue is **team shape**. The team with the ball spreads out to create space; the defending team stays compact. The schematic makes the idea; the real clip after it shows it inside a missing-ball gap.

<img src="figs/fig_width_schematic.png" style="width:100%">

<img src="figs/fig_shape_deform.gif" style="width:100%">

A model trained only on team shape picks the team with the ball correctly 87% of the time on unseen matches, without seeing the ball. That is why the in-gap decoder can recover a hand-over the ball never showed. The catch is coverage: only 3.9% of our graded frames are eligible for the decoder, against 17% to 44% on the practice set, so the expected gain is +.002 to +.005. The clue is real; its reach on this match is small. It also shares a blind spot with the gate — both are weakest at the edge of a stoppage, where the players are just starting to move — so we do not treat their agreement as independent confirmation of a boundary.

## 6. What we tried and dropped

The most tempting idea was to find the kick. A kick is a sudden change in the ball's velocity; on clean public data the textbook detector finds 75% to 83% of them. On our ball it fails, because our ball path is smoothed over about a third of a second, which spreads the kick out. We confirmed the cause by smoothing the practice ball the same way: the method fails there too.

That result set the pattern for the rest of the notebook: a clue that works on clean data is not evidence until it survives our noise. The table below is the honest list, with the number that killed each idea.

| idea | what killed it |
|---|---|
| kicks from ball kinematics | only 75%-83% recall on clean data; our ball is smoothed, so it fails here too |
| ball-player coupling | cost −.002 to −.011 spellF1 when added to the decoder |
| video ball detection | 36% recall on frames where tracking had the ball, 82% false positives on frames where it did not; the ball is 8 px to 10 px |
| explicit-duration HSMM | −.0090 [−.0163, −.0024], negative in nine of ten games |
| restart team by stoppage type | −0.4 points restart accuracy and negative spellF1 in every variant |
| split long stoppage blocks | −.0011, negative in ten of ten games |
| crowd loudness | lead +.027, below the five-point floor |
| public audio-event model | synthetic whistle .115, this match's whistle .001 |

The duration model deserves a sentence of its own because it was the most expensive attempt. An explicit-duration HSMM over learned emissions scores −.0090 [−.0163, −.0024] against the shipped stack, negative in nine of ten games. A learned sticky HMM without the duration prior scores +.0084 [−.0025, +.0180], which is positive but includes zero and is negative in two games: a tier-2 lever, not a replacement. The duration prior is what loses; the emission calibration is the open question.

Restart-team-by-stoppage-type was falsified as pre-registered. Adding the stoppage type changed restart-team accuracy by −0.4 points and made spellF1 worse in every variant, because most long predicted stoppages bridge several real stoppages and some live play: the team before the block is not the team that last touched the ball. Splitting long blocks looked good on the old practice data, but after the degrader fix it scores −.0011, negative in ten of ten games. Crowd loudness tracked microphone placement, not the match: its lead was +.027, below our five-point floor. A public audio-event model heard a synthetic whistle at .115 but this match's real whistle at about .001; our small hand-built detector stays.

## Limitations

The public sample is five minutes and 17 possessions. We report the shipped entry's score there as .456 spellF1 and .712 macroF1, against .326 and .424 for the organizers' two- and five-second baselines, but it is a direction, not a result: the noise band on a full match is about 0.04, and 17 possessions cannot resolve a small difference. Nothing in this notebook was selected on that sample.

The practice set is about .06 easier than the real match, so its levels are not our levels. It has no cooling breaks and no video-review stops; Metrica over-calls dead ball around advantage play; it contains short sliver gaps our match does not; and its resting-ball recall is much lower than clean data's. The share of stoppages that keep the ball visible on our match is only pinned to 7%-23%. The in-gap decoder is eligible on 3.9% of our graded frames, so its +.002 to +.005 expected gain is near the noise floor.

The biggest weakness is unchanged: we catch only about one in five real possessions well enough to count. About 13.5% of owner changes cannot be resolved from geometry at all, and our hand-overs are slightly early, with a scatter that costs far more than the bias. The score is dominated by the possessions we miss, not by the frames we label. The shipped file calls 45.0% of graded frames dead, which is high against a football prior; part of that is a real cooling break and part is the residue of present-ball stoppages we still miss. The short-spell floor is measured but not wired, so the current CSV does not yet contain its gain.

Finally, every figure that renders tracking is local-only under D-002. The public profile of this notebook replaces those figures with placeholders; the prose and the synthetic schematics are the publishable part. The full evidence, including the numbers this notebook does not quote, is in `research/` and `solution/docs/WHAT_WE_TRIED.md`.

## Code, data and acknowledgements

The shipped build is one command in `solution/`: `python -m possession.make_submission`. The pipeline modules are `baseline_plus.py`, `dgate.py`, `ingap.py`, `growd.py`, `subgap.py` and `postprocess.py`; the scoring harness is `dev_v1.py`; the clip renderer is `render.py`. This notebook is built by `scripts/walkthrough_numbers.py` and `scripts/build_walkthrough.py`, and the numbers manifest is `notebooks/numbers.json`.

The practice data is public tracking from the German league (DFL), Metrica and SkillCorner, used under their licenses and never redistributed here. The match data is the organizer's and stays on this machine. The scoring code is the organizer's `evaluation/possession_metrics.py`, imported rather than re-implemented.

Thanks to the organizers for a problem where the honest answer is usually a negative result, and to the four reference notebooks that showed how to write one down.

## Appendix: traceability

Every measured number in the prose is a key in `notebooks/numbers.json`. The table is generated from that file, so it cannot drift from the notebook: the *source* is where the value lives, the *command* is how to see it again, and *recomputed* says whether `--check` rebuilds it from the live pipeline or reads it from a digest. The section column is the part of the notebook that uses the key.

| figure | section | profile | built by | what it shows |
|---|---|---|---|---|
| fig_gate_instruments | 5 | both | `fig_gate_instruments` / `build_walkthrough.py --figs-only --only fig_gate_instruments` | Two independent instruments agree on how much of the missing time is dead. Aggregates only. |
| fig_match_gap | 1 | local only | `_local_match_gap` / `build_walkthrough.py --figs-only --only fig_match_gap` | Our match: live play, then the tracking loses the ball and the label turns D. Local render. |
| fig_missingness | 3 | both | `fig_missingness` / `build_walkthrough.py --figs-only --only fig_missingness` | Where the ball is missing: our match against the degraded practice set. Aggregates only. |
| fig_nearest_player | 1 | both | `fig_nearest_player` / `build_walkthrough.py --figs-only --only fig_nearest_player` | Why the nearest player is not the owner: the ball is in flight, and the nearest player has not touched it. Synthetic. |
| fig_practice_clean | 3 | local only | `_practice_pair` / `build_walkthrough.py --figs-only --only fig_practice_clean` | The practice clip before degrading: the ball stays visible through the stoppage. Local render. |
| fig_practice_degraded | 3 | local only | `_practice_pair` / `build_walkthrough.py --figs-only --only fig_practice_degraded` | The same clip after degrading: the ball is missing where our tracker loses it. Local render. |
| fig_shape_deform | 5 | local only | `_shape_deform` / `build_walkthrough.py --figs-only --only fig_shape_deform` | The shape clue in a real missing-ball gap: the wide team changes as the owner changes. Local render. |
| fig_spellf1 | 2 | both | `fig_spellf1` / `build_walkthrough.py --figs-only --only fig_spellf1` | The centerpiece: a true ten-second possession, a perfect answer, the same answer split, and a missed spell, scored by the organizer's metric. Synthetic. |
| fig_spellf1_curve | 2 | both | `fig_spellf1_curve` / `build_walkthrough.py --figs-only --only fig_spellf1_curve` | The same experiment as a curve: score against the number of fragments. Synthetic. |
| fig_stage_baseline_plus | 4 | local only | `_stage_figs` / `build_walkthrough.py --figs-only --only fig_stage_baseline_plus` | Stage 2, Baseline+: hysteresis, merge and a 2.5 m radius. Local render. |
| fig_stage_gate | 4 | local only | `_stage_figs` / `build_walkthrough.py --figs-only --only fig_stage_gate` | Stage 3, the missing-ball gate: the ball is gone and the players are still, so the stretch becomes D. Local render. |
| fig_stage_grown | 4 | local only | `_stage_figs` / `build_walkthrough.py --figs-only --only fig_stage_grown` | Stage 5, grown stoppages: each resting-ball moment is extended to the stoppage around it. Local render. |
| fig_stage_ingap | 4 | local only | `_stage_figs` / `build_walkthrough.py --figs-only --only fig_stage_ingap` | Stage 4, the in-gap decoder: team shape inserts a hand-over inside a missing-ball gap. Local render. |
| fig_stage_smooth50 | 4 | local only | `_stage_figs` / `build_walkthrough.py --figs-only --only fig_stage_smooth50` | Stage 1, the organizers' 2-second baseline: nearest player, majority-smoothed. Local render. |
| fig_stage_subgap | 4 | local only | `_stage_figs` / `build_walkthrough.py --figs-only --only fig_stage_subgap` | Stage 6, the sub-gap rule: a stoppage inside a gap the main gate left alone. Local render. |
| fig_three_labels | 1 | both | `fig_three_labels` / `build_walkthrough.py --figs-only --only fig_three_labels` | The three labels, and the 25 answers per second they ask for. Synthetic. |
| fig_width_schematic | 5 | both | `fig_width_schematic` / `build_walkthrough.py --figs-only --only fig_width_schematic` | The team with the ball is wider; the defending team is narrow. Synthetic. |

| key | value | section | source | command | recomputed |
|---|---|---|---|---|---|
| all_frames | 199,319 | 1 | research/Q26_e66_repro.md (validator) | `grep -n -m1 '199,319 rows' research/Q26_e66_repro.md` | no |
| ball_in_play_min | 58.3 min | 4 | research/Q26_e66_repro.md (validator counts) | `grep -n -m1 'H 51,037' research/Q26_e66_repro.md` | no |
| ball_px | 8 px to 10 px | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'ball is (8) to 10 pixels wide' solution/docs/WHAT_WE_TRIED.md` | no |
| baseline_plus_dev | .3259 | 4 | solution/possession/dev_v1.py (reference_results) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| baseline_plus_new | .326 | 3 | work/FINDINGS.md | `grep -n -m1 'Baseline\+ \.216 → (\.326)' work/FINDINGS.md` | no |
| baseline_plus_old | .216 | 3 | work/FINDINGS.md | `grep -n -m1 '\*\*Baseline\+ (\.216) → \.326' work/FINDINGS.md` | no |
| baseline_plus_over_smooth50 | +.0222 [+.0058, +.0403] | 4 | solution/possession/dev_v1.py (compare_dev_v1) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| coupling_high | −.011 | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'cost −0\.002 to −(0\.011)' solution/docs/WHAT_WE_TRIED.md` | no |
| coupling_low | −.002 | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'cost −(0\.002) to −0\.011' solution/docs/WHAT_WE_TRIED.md` | no |
| crowd_lead | +.027 | 6 | work/FINDINGS.md | `grep -n -m1 'crowd parks\*\* \(lead (\+\.027)' work/FINDINGS.md` | no |
| csv_sha8 | 8255d2d1…5e61 | 4 | research/Q26_e66_repro.md (Q26 clean-clone build) | `grep -n -m1 'sha256' research/Q26_e66_repro.md` | no |
| demo_iou2 | 0.50 | 2 | synthetic labels (one true 250-frame spell) | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_iou2` | yes |
| demo_iou3 | 0.33 | 2 | synthetic labels (one true 250-frame spell) | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_iou3` | yes |
| demo_iou4 | 0.25 | 2 | synthetic labels (one true 250-frame spell) | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_iou4` | yes |
| demo_iou_high | 0.7 | 2 | the organizer's spellF1 band (SPELL_IOUS) | `grep -n 'SPELL_IOUS' more-than-a-hack-2027/evaluation/possession_metrics.py` | no |
| demo_iou_low | 0.3 | 2 | the organizer's spellF1 band (SPELL_IOUS) | `grep -n 'SPELL_IOUS' more-than-a-hack-2027/evaluation/possession_metrics.py` | no |
| demo_missed | .00 | 2 | organizer metric, synthetic labels | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_missed` | yes |
| demo_perfect | 1.00 | 2 | organizer metric, synthetic labels | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_perfect` | yes |
| demo_split2 | .40 | 2 | organizer metric, synthetic labels | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_split2` | yes |
| demo_split3 | .10 | 2 | organizer metric, synthetic labels | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_split3` | yes |
| demo_split4 | .00 | 2 | organizer metric, synthetic labels | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show demo_split4` | yes |
| e25_ci_high | 89.2% | 5 | work/FINDINGS.md | `grep -n -m1 '85\.6% \[80\.8, (89\.2)\]' work/FINDINGS.md` | no |
| e25_ci_low | 80.8% | 5 | work/FINDINGS.md | `grep -n -m1 '85\.6% \[(80\.8), 89\.2\]' work/FINDINGS.md` | no |
| e25_dead_share | 85.6% | 5 | work/FINDINGS.md | `grep -n -m1 'flags \*\*(85\.6%)' work/FINDINGS.md` | no |
| e36_dead_share | 84.5% | 5 | work/FINDINGS.md | `grep -n -m1 'independent route puts \*\*(84\.5%)' work/FINDINGS.md` | no |
| eligible_dev | 17% to 44% | 5 | work/FINDINGS.md | `grep -n -m1 'vs (17)-44%' work/FINDINGS.md` | no |
| eligible_ours | 3.9% | 4, 5, limits | work/FINDINGS.md | `grep -n -m1 'eligible pool (3\.9%)' work/FINDINGS.md` | no |
| expected_gate | +.06 to +.08 | 4 | research/E76_regrid.md | `grep -n -m1 'D gate: (\+\.06)\.\.\+\.08' research/E76_regrid.md` | no |
| expected_grown | +.001 to +.005 | 4 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'We expect (\+\.001) to \+\.005 on our match' solution/docs/WHAT_WE_TRIED.md` | no |
| expected_ingap | +.002 to +.005 | 4, 5, limits | research/E76_regrid.md | `grep -n -m1 'Ingap: ~ (\+\.002)\.\.\+\.005' research/E76_regrid.md` | no |
| expected_postprocess | +.006 to +.009 | 4 | work/FINDINGS.md | `grep -n -m1 'expected (\+\.006)\.\.\+\.009' work/FINDINGS.md` | no |
| expected_subgap | +.002 | 4 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'We expect about (\+\.002) on our match' solution/docs/WHAT_WE_TRIED.md` | no |
| flaw_target_auc | .80 | 3 | work/FINDINGS.md | `grep -n -m1 'target (\.80) not met' work/FINDINGS.md` | no |
| flaw_texture_auc | .92 | 3 | work/FINDINGS.md | `grep -n -m1 'texture-only LOGO 1\.000 → (\.92)' work/FINDINGS.md` | no |
| flaw_texture_auc_heldout | .907 | 3 | work/FINDINGS.md | `grep -n -m1 'give texture (\.907)' work/FINDINGS.md` | no |
| flip_recall_clean | .764 | 1 | work/FINDINGS.md | `grep -n -m1 'nearest-player cue catches \*\*(\.764)\*\*' work/FINDINGS.md` | no |
| flip_recall_degraded | .482 | 1 | work/FINDINGS.md | `grep -n -m1 'vs \*\*(\.482)\*\* on the degraded copy' work/FINDINGS.md` | no |
| fps | 25 Hz | 1 | solution/possession/evaluate.py: FPS | `grep -n '^FPS' solution/possession/evaluate.py` | yes |
| gate_dev | .3987 | 4 | solution/possession/dev_v1.py (reference_results) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| gate_over_baseline_plus | +.0728 [+.0649, +.0808] | 4 | solution/possession/dev_v1.py (compare_dev_v1) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| gate_pd_missing_high | .99 | 5 | work/FINDINGS.md | `grep -n -m1 'corrected P\(D\\|missing\) \*\*0\.93-(0\.99)' work/FINDINGS.md` | no |
| gate_pd_missing_low | .93 | 5 | work/FINDINGS.md | `grep -n -m1 'corrected P\(D\\|missing\) \*\*(0\.93)' work/FINDINGS.md` | no |
| graded_frames | 158,857 | 1 | research/Q26_e66_repro.md (validator) | `grep -n -m1 'H 51,037' research/Q26_e66_repro.md` | no |
| grown_ci_high | .0066 | 4 | research/E76_regrid.md | `grep -n -m1 'R25 cap30 \\| \+\.0055 \[\+\.0159\] \\| \[\.0044,(\.0066)\]' research/E76_regrid.md` | no |
| grown_ci_low | .0044 | 4 | research/E76_regrid.md | `grep -n -m1 'R25 cap30 \\| \+\.0055 \[\+\.0159\] \\| \[(\.0044),' research/E76_regrid.md` | no |
| grown_delta | +.0055 | 4 | research/E76_regrid.md | `grep -n -m1 'E65 grown D, R25 cap30 \\| (\+\.0055)' research/E76_regrid.md` | no |
| hmm_ci_high | +.0180 | 6 | work/FINDINGS.md | `grep -n -m1 '\+\.0084 \[−\.0025,(\+\.0180)\]' work/FINDINGS.md` | no |
| hmm_ci_low | −.0025 | 6 | work/FINDINGS.md | `grep -n -m1 '\+\.0084 \[(−\.0025),\+\.0180\]' work/FINDINGS.md` | no |
| hmm_delta | +.0084 | 6 | work/FINDINGS.md | `grep -n -m1 'HMM (\.4123), \+\.0084' work/FINDINGS.md` | no |
| hsmm_ci_high | −.0024 | 6 | work/FINDINGS.md | `grep -n -m1 '−\.0090 \[−\.0163,(−\.0024)\]' work/FINDINGS.md` | no |
| hsmm_ci_low | −.0163 | 6 | work/FINDINGS.md | `grep -n -m1 '−\.0090 \[(−\.0163),−\.0024\]' work/FINDINGS.md` | no |
| hsmm_delta | −.0090 | 6 | work/FINDINGS.md | `grep -n -m1 '\*\*HSMM (\.3949), −\.0090' work/FINDINGS.md` | no |
| ingap_new | +.006 | 3 | work/FINDINGS.md | `grep -n -m1 'in-gap decoder \+\.0816 → (\+\.006)' work/FINDINGS.md` | no |
| ingap_old | +.082 | 3 | work/FINDINGS.md | `grep -n -m1 'in-gap decoder \+\.082 → (\+\.003\.\.\+\.016)' work/FINDINGS.md` | no |
| ingap_over_gate | +.0052 [+.0038, +.0065] | 4 | solution/possession/dev_v1.py (compare_dev_v1) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| kick_recall_high | 83% | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'Recall 75 to (83%) on clean data' solution/docs/WHAT_WE_TRIED.md` | no |
| kick_recall_low | 75% | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'Recall (75) to 83% on clean data' solution/docs/WHAT_WE_TRIED.md` | no |
| missing_share_graded | 40.7% | 1, 3 | solution/possession/data.py (ball channel) | `cd solution && .venv/bin/python scripts/walkthrough_numbers.py --show missing_share_graded` | yes |
| n_labels | three | 1 | work/BIGPICTURE.md | `grep -n 'three things' work/BIGPICTURE.md` | no |
| nearest_radius_m | 2.5 m | 1, 4 | solution/possession/baseline_plus.py DEFAULT_PARAMS | `grep -n 'DEFAULT_PARAMS' solution/possession/baseline_plus.py` | no |
| noise_band | 0.04 | 2, limits | solution/docs/GLOSSARY.md | `grep -n -m1 'below about (0\.04) on the full match' solution/docs/GLOSSARY.md` | no |
| ours_live_missing | ~3% | 3 | work/FINDINGS.md | `grep -n -m1 'P\(NaN\\|live\) \.43 → \.044 \(~(\.03)\)' work/FINDINGS.md` | no |
| ours_long_gap_live | ≤ ~4% | 3 | work/FINDINGS.md | `grep -n -m1 'live share dev 35-39% vs (≤ ~4)%' work/FINDINGS.md` | no |
| panns_real | .001 | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'real whistles at about (\.001)' solution/docs/WHAT_WE_TRIED.md` | no |
| panns_synthetic | .115 | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'synthetic whistle tone at (\.115)' solution/docs/WHAT_WE_TRIED.md` | no |
| postprocess_ci_high | .0138 | 4 | research/E14_result.md | `grep -n -m1 '\[\+\.0063, \+(\.0138)\]' research/E14_result.md` | no |
| postprocess_ci_low | .0063 | 4 | research/E14_result.md | `grep -n -m1 '\[\+(\.0063), \+\.0138\]' research/E14_result.md` | no |
| postprocess_delta | +.0099 | 4 | research/E14_result.md | `grep -n -m1 '(\+\.0099) spellF1 \[\+\.0063, \+\.0138\]' research/E14_result.md` | no |
| practice_easier | .06 | 3, limits | solution/docs/HOW_WE_WORK.md | `grep -n -m1 'about (\.06) easier' solution/docs/HOW_WE_WORK.md` | no |
| practice_games | 10 | 3 | work/DECISIONS.md D-022 | `grep -n '10 games' work/DECISIONS.md` | no |
| practice_idsse | seven | 3 | work/DECISIONS.md D-022 | `grep -n '7 IDSSE' work/DECISIONS.md` | no |
| practice_live_missing | 4.4% | 3 | work/FINDINGS.md | `grep -n -m1 'P\(NaN\\|live\) \.43 → (\.044)' work/FINDINGS.md` | no |
| practice_live_missing_old | 42% | 3 | work/FINDINGS.md | `grep -n -m1 'P\(ball missing \\| live\) dev (\.42)' work/FINDINGS.md` | no |
| practice_long_gap_live | 2.1% | 3 | work/FINDINGS.md | `grep -n -m1 'long-gap live share \.35-\.39 → (\.021)' work/FINDINGS.md` | no |
| practice_metrica | three | 3 | work/DECISIONS.md D-022 | `grep -n '3 Metrica' work/DECISIONS.md` | no |
| practice_missing_share | 37.3% | 3 | research/E76_regrid.md | `grep -n -m1 'missing share (\.373) at primary' research/E76_regrid.md` | no |
| pred_d_runs | 114 | 4 | work/FINDINGS.md (E14) | `grep -n -m1 'D stretches 114' work/FINDINGS.md` | no |
| pred_d_share_graded | 45.0% | 4, limits | research/Q26_e66_repro.md (validator counts) | `grep -n -m1 'H 51,037' research/Q26_e66_repro.md` | no |
| pred_spells | 414 | 4 | research/Q26_e66_repro.md | `grep -n -m1 'Final spells 414' research/Q26_e66_repro.md` | no |
| public_baseline2 | .326 | limits | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'against (\.326) for their 2-second baseline' solution/docs/WHAT_WE_TRIED.md` | no |
| public_baseline5 | .424 | limits | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'and (\.424) for their 5-second one' solution/docs/WHAT_WE_TRIED.md` | no |
| public_gt_spells | 17 | 2, 3, limits | work/DECISIONS.md D-013 | `grep -n '17 GT spells' work/DECISIONS.md` | no |
| public_minutes | five | 3, limits | work/GUARDRAILS.md §2 | `grep -n 'public 5 minutes' work/GUARDRAILS.md` | no |
| public_ship_macro | .712 | limits | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'Our frame-level score there is (\.712)' solution/docs/WHAT_WE_TRIED.md` | no |
| public_ship_spellf1 | .456 | limits | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'the shipped entry scores (\.456)' solution/docs/WHAT_WE_TRIED.md` | no |
| restart_team_pp | −0.4 points | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'changed it by (−0\.4) points' solution/docs/WHAT_WE_TRIED.md` | no |
| shape_correct | 87% | 5 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'Correct (87%) of the time' solution/docs/WHAT_WE_TRIED.md` | no |
| ship_dev | .4039 | 4 | solution/possession/dev_v1.py (reference_results) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| ship_over_smooth50 | +.1003 [+.0877, +.1130] | 4 | solution/possession/dev_v1.py (compare_dev_v1) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| skillcorner_games | 20 | 3 | work/DECISIONS.md D-022 | `grep -n '20 games' work/DECISIONS.md` | no |
| smooth50_dev | .3037 | 3, 4 | solution/possession/dev_v1.py (reference_results) | `cd solution && .venv/bin/python -c "from possession import dev_v1; r=dev_v1.reference_results(); print({k: round(v.summary()['headline'],4) for k,v in r.items()})"` | yes |
| smooth50_match | .243 | 3 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'against (\.243) on the real match' solution/docs/WHAT_WE_TRIED.md` | no |
| split_delta | −.0011 | 6 | research/E76_regrid.md | `grep -n -m1 'E72 split D blocks \\| (-?\.0011)' research/E76_regrid.md` | no |
| subgap_ci_high | .0056 | 4 | research/E76_regrid.md | `grep -n -m1 'E69 sub-gap gate overlay \\| \+\.0044 \[\+\.0032\] \\| \[\.0033,(\.0056)\]' research/E76_regrid.md` | no |
| subgap_ci_low | .0033 | 4 | research/E76_regrid.md | `grep -n -m1 'E69 sub-gap gate overlay \\| \+\.0044 \[\+\.0032\] \\| \[(\.0033),' research/E76_regrid.md` | no |
| subgap_delta | +.0044 | 4 | research/E76_regrid.md | `grep -n -m1 'E69 sub-gap gate overlay \\| (\+\.0044)' research/E76_regrid.md` | no |
| unresolvable_share | 13.5% | 1, limits | work/FINDINGS.md | `grep -n -m1 '~(13\.5%) of all flips unresolvable' work/FINDINGS.md` | no |
| video_false | 82% | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'fired on (82%) of frames' solution/docs/WHAT_WE_TRIED.md` | no |
| video_recall | 36% | 6 | solution/docs/WHAT_WE_TRIED.md | `grep -n -m1 'found it on (36%) of frames' solution/docs/WHAT_WE_TRIED.md` | no |
| visible_share_hi | 23% | 3, limits | solution/docs/HOW_WE_WORK.md | `grep -n -m1 'between 7% and (23%) of stoppages' solution/docs/HOW_WE_WORK.md` | no |
| visible_share_lo | 7% | 3, limits | solution/docs/HOW_WE_WORK.md | `grep -n -m1 'between (7%) and 23% of stoppages' solution/docs/HOW_WE_WORK.md` | no |